# NB91 - P8 ablation aggregator

Reads the per-epoch `results.csv` for each of the 3 ablation rows from
`/content/drive/MyDrive/EcoCAR/training_runs/checkpoints/<row>/` and
builds the final ablation table.

Run this notebook AFTER NB88, NB89, NB90 have produced (at least one
epoch of) results. You can run it at any point during training to
preview the current standings - just re-run the cell.


### Cell 1: Mount Drive

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive

if not Path('/content/drive').exists():
    drive.mount('/content/drive', force_remount=False)

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
print('[ok] mounted')


### Cell 2: Read per-row results.csv + print the table

Picks the FINAL (last) row of each results.csv. If a run is still in
progress, this shows the latest epoch's metrics. Writes
`P8_train/results.md` so the table is git-trackable.


In [ ]:
import csv
from pathlib import Path

ROWS = ['clr_lane_default', 'clr_lane_no_square_priors', 'clr_lane_no_gca']
CKPT_ROOT = Path('/content/drive/MyDrive/EcoCAR/training_runs/checkpoints')

def _final_row(csv_path):
    if not csv_path.exists():
        return None
    with csv_path.open() as f:
        rows = list(csv.DictReader(f))
    return rows[-1] if rows else None

def _g(row, *candidates):
    if row is None:
        return None
    for key_pat in candidates:
        for ck in row:
            if ck.strip().lower() == key_pat.lower():
                try:
                    return float(row[ck])
                except (TypeError, ValueError):
                    return row[ck]
    return None

records = []
for name in ROWS:
    r = _final_row(CKPT_ROOT / name / 'results.csv')
    records.append({
        'name': name,
        'epoch': _g(r, 'epoch'),
        'mAP50':    _g(r, 'metrics/mAP50(B)', 'metrics/mAP50', 'mAP50'),
        'mAP50-95': _g(r, 'metrics/mAP50-95(B)', 'metrics/mAP50-95'),
        'P(det)':   _g(r, 'metrics/precision(B)', 'metrics/precision'),
        'R(det)':   _g(r, 'metrics/recall(B)', 'metrics/recall'),
        'IoU(lane)': _g(r, 'metrics/IoU(lane)'),
        'mIoU(lane)': _g(r, 'metrics/mIoU(lane)'),
        'pixacc(lane)': _g(r, 'metrics/pixacc(lane)'),
        'subacc(lane)': _g(r, 'metrics/subacc(lane)'),
    })

lines = ['# P8 ablation table\n']
lines.append('Pulled from `/content/drive/MyDrive/EcoCAR/training_runs/checkpoints/<row>/results.csv` '
             '(end-of-last-completed-epoch).\n')
lines.append('| Row | Epoch | mAP50 | mAP50-95 | P(det) | R(det) | IoU(lane) | mIoU | pixAcc | subAcc |')
lines.append('|-----|-------|-------|----------|--------|--------|-----------|------|--------|--------|')
def _fmt(x):
    if x is None: return 'TBD'
    if isinstance(x, float): return f'{x:.4f}'
    return str(x)
for rec in records:
    cells = [rec['name']] + [_fmt(rec[k]) for k in ('epoch', 'mAP50', 'mAP50-95',
                                                   'P(det)', 'R(det)',
                                                   'IoU(lane)', 'mIoU(lane)',
                                                   'pixacc(lane)', 'subacc(lane)')]
    lines.append('| ' + ' | '.join(f'`{cells[0]}`' if i == 0 else c for i, c in enumerate(cells)) + ' |')

print('\n'.join(lines))

OUT = Path(REPO_ROOT) / 'stage2/rmt_ppad_migration/P8_train/results.md'
OUT.write_text('\n'.join(lines), encoding='utf-8')
print(f'\n[wrote] {OUT}')
